### Firewall API - Network Interfaces Management

[![Python 3.11+](https://img.shields.io/badge/Python-3.11%2B-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](../LICENSE)
[![Development Status: Beta 0.8.0](https://img.shields.io/badge/Status-Beta%200.8.0-orange.svg)](https://pypi.org/project/sfos-api/)
[![Code Style: Ruff](https://img.shields.io/endpoint?url=https://raw.githubusercontent.com/astral-sh/ruff/main/assets/badge/v2.json)](https://github.com/astral-sh/ruff)

**Purpose:** Create, read, update, and delete network interface objects (such as VLAN interfaces) and inspect existing interfaces.

**Topics:**
- Basic CRUD operations using domain StrEnum objects (`Network`)
- Managing network interfaces (`Network.INTERFACES_NETWORK_INTERFACES`)
- Filtering and auto-pagination
- Error handling

**Requirements:**
- Credentials configured (`.credentials/config.ini`)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [ ]:
from utils_api import find_credentials_file, print_data

from sfos_api import Network, SFOSClient

# Initialize the SFOS API Client
client = SFOSClient(config_path=find_credentials_file())

#### Create Interface

Create a network interface (`vlan`, `bridge`, or `lag`) using the `Network.INTERFACES_NETWORK_INTERFACES` endpoint object. Physical, XFRM, and cellular WAN interfaces cannot be created via the API.

API Link: [Create an interface | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/create-network-interface/)

---

In [ ]:
# Create a VLAN interface using Endpoint object
endpoint = Network.INTERFACES_NETWORK_INTERFACES
interface_name = "test-vlan-100"

payload = {
    "name": interface_name,
    "description": "This is created by SFOS API",
    "type": "vlan",
    "vlanDetails": {
        "vlanId": 100,
        "parentInterface": {"name": "Port1"},
    },
    "zone": {"name": "LAN"},
    "ipv4": {
        "assignmentType": "static",
        "ipv4Address": "198.51.100.1",
        "cidr": 24,
    },
}

try:
    response = client.post(endpoint, payload)
    print_data(response)
except Exception as e:
    print(f"Error creating interface: {e}")
    print_data(str(e))

#### Get Interface

Retrieve a specific network interface object by ID or name.

API Link: [Get an interface | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-network-interface/)

---

In [ ]:
# Get specific interface by name or ID
endpoint = Network.INTERFACES_NETWORK_INTERFACES
id_or_name = "test-vlan-100"

try:
    response = client.get(endpoint, id_or_name=id_or_name)
    print_data(response)
except Exception as e:
    print(f"Error reading interface: {e}")
    print_data(str(e))

#### Get All Interfaces

Retrieve all network interface objects from the firewall with automatic pagination, filtering, and manual pagination options.

API Link: [Get all interfaces | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/list-network-interfaces/)

---

In [ ]:
# Get all interfaces (auto-paginated)
endpoint = Network.INTERFACES_NETWORK_INTERFACES

try:
    response = client.get(endpoint)
    if "items" in response:
        print(f"✅ Total interfaces retrieved: {len(response['items'])}")
        print_data(
            response["items"],
            include=["name", "type", "hardwareName", "description"],
            numbering=True,
        )
    else:
        print_data(response)
except Exception as e:
    print(f"Error reading interfaces: {e}")
    print_data(str(e))

In [ ]:
# Filter interfaces by name (auto-paginated)
endpoint = Network.INTERFACES_NETWORK_INTERFACES

try:
    response = client.get(endpoint, nameContains="Port")
    if "items" in response:
        print(f"✅ Matching interfaces: {len(response['items'])}")
        print_data(response["items"], include=["name", "type", "hardwareName"])
    else:
        print_data(response)
except Exception as e:
    print(f"Error filtering interfaces: {e}")
    print_data(str(e))

In [ ]:
# Retrieve a single page only (no auto-pagination when page is specified)
endpoint = Network.INTERFACES_NETWORK_INTERFACES

try:
    response = client.get(endpoint, page=1, pageSize=5)
    if "items" in response:
        pages_info = response.get("pages", {})
        current_page = pages_info.get("current", response.get("page", 1))
        print(f"✅ Page {current_page} only: {len(response['items'])} items on this page")
        print_data(response["items"])
    else:
        print_data(response)
except Exception as e:
    print(f"Error reading interfaces: {e}")
    print_data(str(e))

In [ ]:
# Keyword arguments take precedence over params dict
endpoint = Network.INTERFACES_NETWORK_INTERFACES
params = {
    "page": 1,
    "pageSize": 50,
    "nameContains": "test-",
}

try:
    response = client.get(endpoint, params=params, pageSize=5)
    if "items" in response:
        pages_info = response.get("pages", {})
        current_page = pages_info.get("current", response.get("page", 1))
        print(f"✅ Retrieved page {current_page} only (not auto-paginated)")
        print_data(response["items"])
    else:
        print_data(response)
except Exception as e:
    print(f"Error reading interfaces: {e}")
    print_data(str(e))

#### Update Interface

Modify an existing network interface object's properties.

API Link: [Update an interface | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/update-network-interface/)

---

In [ ]:
# Update interface
endpoint = Network.INTERFACES_NETWORK_INTERFACES
id_or_name = "test-vlan-100"

payload = {
    "description": "Updated by SFOS API",
    "mtu": 1500,
}

try:
    response = client.patch(endpoint, id_or_name=id_or_name, data=payload)
    print(f"✅ Updated interface '{id_or_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating interface: {e}")
    print_data(str(e))

#### Delete Interface

Remove a network interface object by ID or name.

API Link: [Delete an interface | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/delete-network-interface/)

---

In [ ]:
# Delete a single interface
endpoint = Network.INTERFACES_NETWORK_INTERFACES
id_or_name = "test-vlan-100"

try:
    response = client.delete(endpoint, id_or_name=id_or_name)
    print(f"✅ Deleted interface '{id_or_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting interface: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()